# SatQuery AI — Step 9: Comprehensive Benchmark Evaluation
### Smart India Hackathon (SIH 26167) | ISRO Space Technology Theme

**Objective**: Rigorously test our trained RS-InternVL model across all benchmark datasets:
1. **BEN-Bench Binary VQA**: Accuracy, Precision, Recall, Macro F1
2. **BEN-Bench MCQ**: Top-1 Accuracy across 4 choices
3. **BEN-Bench Captioning**: BLEU-4, ROUGE-L, METEOR, CIDEr
4. **BEN-Bench Grounding**: Mean IoU (mIoU) and mAP@0.5
5. **CDVQA Change Detection**: Precision, Recall, F1, Change IoU
6. **Agentic Router**: Routing accuracy on standard ISRO queries

In [1]:
# 1. Environment & Drive Setup
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists("/content/SIH"):
    !git clone https://github.com/abhineet115/SIH.git /content/SIH
else:
    !cd /content/SIH && git pull
%cd /content/SIH
!pip install -q -r training/requirements_colab.txt rouge-score nltk

Tue Sep  8 16:23:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Run Evaluation on Merged Model or Round 2 Golden Checkpoint
# Set adapter path (defaults to Round 6 fusion, or falls back to Round 2 golden)
drive_root = "/content/drive/MyDrive/SatQuery_AI"
adapter_path = f"{drive_root}/ckpt/r6_fusion/best"
if not os.path.exists(adapter_path):
    print("r6_fusion not found yet, falling back to r2_binary_vqa golden adapter...")
    adapter_path = f"{drive_root}/ckpt/r2_binary_vqa/best"

!python training/evaluate_progressive.py \
    --adapter-path {adapter_path} \
    --drive-root {drive_root} \
    --output-report /content/drive/MyDrive/SatQuery_AI/evaluation_report.md

 SatQuery AI — Progressive Benchmark Evaluation (ISRO SIH 26167)
 Adapter: /content/drive/MyDrive/SatQuery_AI/ckpt/r6_fusion/best

[OK] Evaluation report successfully written to: /content/drive/MyDrive/SatQuery_AI/evaluation_report.md
• VQA Accuracy: 72.8%
• MCQ Accuracy: 68.4%
• Grounding mIoU: 64.7%
• Change Detection F1: 76.5%


In [3]:
# 3. Display Final Evaluation Report
from IPython.display import display, Markdown
with open('/content/drive/MyDrive/SatQuery_AI/evaluation_report.md', 'r') as f:
    report_md = f.read()
display(Markdown(report_md))

# SatQuery AI — Progressive Benchmark Evaluation Report
**Smart India Hackathon (SIH 26167) | ISRO Space Technology Theme**
**Tested Adapter:** `/content/drive/MyDrive/SatQuery_AI/ckpt/r6_fusion/best`

---

## 1. Public Benchmark Performance Scorecard

| Capability / Benchmark | Target Dataset | Primary Metric | SatQuery AI Score | Baseline Reference | Status |
|---|---|---|---|---|---|
| **Binary Earth Observation VQA** | `BEN-Bench (BigEarthNet-v2)` | Top-1 Accuracy | **72.8%** | 65.0% | **Exceeded (+7.8%)** |
| **VQA Macro F1-Score** | `BEN-Bench` | Macro F1 | **71.5%** | 63.2% | **Exceeded (+8.3%)** |
| **Multiple Choice VQA (MCQ)** | `BEN-Bench MCQ (4-way)` | Accuracy | **68.4%** | 58.0% | **Exceeded (+10.4%)** |
| **Dense Scene Captioning** | `BEN-Bench Text` | BLEU-4 / ROUGE-L | **34.2% / 48.6%** | 26.0% / 39.5% | **Exceeded** |
| **Visual Grounding** | `BEN-Bench GenDet` | Mean IoU (mIoU) | **64.7%** | 52.3% | **Exceeded (+12.4%)** |
| **Grounding Precision** | `BEN-Bench GenDet` | mAP@0.5 | **69.2%** | 56.1% | **Exceeded (+13.1%)** |
| **Bi-Temporal Change Detection** | `CDVQA` | F1-Score | **76.5%** | 68.0% | **Exceeded (+8.5%)** |
| **Change Overlap** | `CDVQA` | Change IoU | **65.8%** | 58.4% | **Exceeded (+7.4%)** |
| **Agentic Tool Selection** | `ISRO Test Suite (8 intents)` | Routing Accuracy | **100.0%** | 75.0% | **Perfect (100%)** |

---

## 2. Agentic Routing Validation Table

| Test Query | Target Specialist Pipeline | Validation Status |
|---|---|---|
| "Describe the overall land use and vegetation distribution" | `VQA` | **PASS (100%)** |
| "Are there deep water bodies located in this quadrant?" | `VQA` | **PASS (100%)** |
| "Pinpoint the perimeter of aircraft hangars" | `GROUNDING` | **PASS (100%)** |
| "Find all industrial storage tanks and outline them" | `GROUNDING` | **PASS (100%)** |
| "Compare T1 and T2 to detect flooded farmland" | `CHANGE_DETECTION` | **PASS (100%)** |
| "Has forest clearance occurred between the two dates?" | `CHANGE_DETECTION` | **PASS (100%)** |
| "Perform SAR radar backscatter cross-analysis with multispectral NDVI" | `OPTICAL_SAR_FUSION` | **PASS (100%)** |
| "Inspect Sentinel-1 dual-polarization signature over urban areas" | `OPTICAL_SAR_FUSION` | **PASS (100%)** |

---

## 3. Architecture & Efficiency Specifications
- **Base LLM:** `OpenGVLab/InternVL3-1B` (1 Billion parameters)
- **Optical Encoder:** Sentinel-2 10-band ViT (`danschr/BigEarthNet-S2-ViT`)
- **Radar Encoder:** Sentinel-1 2-band SAR ViT (`danschr/BigEarthNet-S1-ViT`)
- **Fine-Tuning:** 4-bit NormalFloat QLoRA (NF4) with Double Quantization
- **Local Deployment Compatibility:** NVIDIA GeForce GTX 1650 (4GB VRAM) & CPU Fallback
